# Глава 2. Индексация данных для RAG

**Задача:** подготовить базу знаний учебного центра и увидеть каждый этап. Notebook использует TXT/Markdown из `data/`; можно добавить PDF с текстовым слоем.

По умолчанию `DEMO=True`: только хеш-векторы слов без API. Для настоящих семантических эмбеддингов настройте `.env` и поставьте `DEMO=False`. Чат-модель нужна только для дополнительного примера в конце.

Запускайте ячейки последовательно. Для повторного опыта с другой моделью или splitter используйте новую папку индекса.

In [1]:
from pathlib import Path
import os

# Jupyter обычно открывается либо в корне проекта, либо в notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data").is_dir(), "Запустите Jupyter из каталога проекта"
os.chdir(ROOT)

from chapter2.pipeline import (
    Settings, load_documents, split_documents, split_markdown,
    make_embeddings, open_store, synchronize,
)
DEMO = True
settings = Settings.from_env()
print("Режим:", "хеш-векторы слов без API" if DEMO else "реальный провайдер")
print("Размер чанка:", settings.chunk_size, "Перекрытие:", settings.chunk_overlap)

Режим: хеш-векторы слов без API
Размер чанка: 500 Перекрытие: 100


## 1. Файлы превращаются в Document
`page_content` содержит текст, `metadata` — источник и номер страницы PDF (с нуля). Загрузчик не превращает скан в текст: для скана нужен отдельный OCR.

In [2]:
documents = load_documents(ROOT / "data")
print("Документов или страниц:", len(documents))
print(documents[0].metadata)
print(documents[0].page_content[:700])

Документов или страниц: 6
{'source': 'README.md'}
# Учебные данные
Все названия, суммы, даты, правила и контакты в этом каталоге вымышлены.
Этот файл тоже попадёт в индекс: загрузчик читает все .txt, .md и .pdf рекурсивно.



## 2. Разбиение на чанки
Рекурсивный splitter использует границы абзацев и строк, затем более мелкие разделители. Размер задаётся в символах. Для сравнения поменяйте параметры в экспериментальной ячейке; основной индекс использует настройки `.env`.

In [3]:
chunks = split_documents(documents, settings)
print("Чанков:", len(chunks))
for chunk in chunks[:4]:
    print(chunk.metadata, "символов:", len(chunk.page_content))
    print(chunk.page_content, "\n")

Чанков: 9
{'source': 'README.md', 'start_index': 0} символов: 172
# Учебные данные
Все названия, суммы, даты, правила и контакты в этом каталоге вымышлены.
Этот файл тоже попадёт в индекс: загрузчик читает все .txt, .md и .pdf рекурсивно. 

{'source': 'cafeteria.pdf', 'page': 0, 'start_index': 0} символов: 136
Orbita campus cafeteria
Building A. Open weekdays 08:00-18:00.
Student lunch costs 250 RUB.
Fictional classroom sample for a PDF loader. 

{'source': 'exams.md', 'start_index': 0} символов: 488
# Экзамены и пересдачи

## Основная сессия
Зимняя экзаменационная сессия проходит с 12 по 25 января. Расписание публикуют в личном кабинете не позднее чем за две недели до первого экзамена. На экзамен нужно принести студенческий билет.

## Пересдача
При неудовлетворительной оценке студент имеет право на две пересдачи. Первую принимает преподаватель, вторую — комиссия из трёх человек. Запись на пересдачу открывается в учебном офисе после окончания основной сессии. Пересдачи бесплатны. 

{'s

In [4]:
from dataclasses import replace
for size, overlap in [(250, 0), (250, 60), (500, 100)]:
    experiment = split_documents(documents, replace(settings, chunk_size=size, chunk_overlap=overlap))
    print(f"size={size}, overlap={overlap}: {len(experiment)} чанков, "
          f"{sum(len(d.page_content) for d in experiment)} символов суммарно")

size=250, overlap=0: 19 чанков, 2663 символов суммарно
size=250, overlap=60: 19 чанков, 2836 символов суммарно
size=500, overlap=100: 9 чанков, 2677 символов суммарно


### Альтернативы: Markdown и исходный код
Эти чанки показываются отдельно и не добавляются в основной индекс.

In [5]:
markdown = (ROOT / "data/scholarships.md").read_text(encoding="utf-8")
for section in split_markdown(markdown):
    print(section.metadata)
    print(section.page_content[:160])

from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
code_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=120, chunk_overlap=20,
)
sample_code = "def square(x):\n    return x * x\n\ndef greet(name):\n    return f'Привет, {name}'\n"
print(code_splitter.split_text(sample_code))

{'title': 'Стипендии учебного центра «Орбита»', 'section': 'Академическая стипендия'}
# Стипендии учебного центра «Орбита»  
## Академическая стипендия
Академическая стипендия составляет 7000 рублей в месяц. Её получают студенты очной формы обуче
{'title': 'Стипендии учебного центра «Орбита»', 'section': 'Социальная поддержка'}
## Социальная поддержка
Социальная стипендия составляет 10000 рублей в месяц и не зависит от оценок. Для оформления нужны заявление и справка о праве на государ
["def square(x):\n    return x * x\n\ndef greet(name):\n    return f'Привет, {name}'"]


### Дополнительно: web-загрузчик
Ячейка отключена по умолчанию. Если включить, она выполнит внешний HTTP-запрос. Сначала извлекается текст HTML; статьи с JavaScript или сложной вёрсткой могут требовать другой загрузчик. В основной индекс этот документ не добавляется.

In [6]:
LOAD_WEB = False
if LOAD_WEB:
    os.environ.setdefault("USER_AGENT", "LangChainChapter2Classroom/1.0")
    from langchain_community.document_loaders import WebBaseLoader
    web_docs = WebBaseLoader("https://example.com").load()
    print(web_docs[0].metadata)
    print(web_docs[0].page_content[:500])

## 3. Текст превращается в векторы
Сравним документ и запрос в одном координатном пространстве. В реальном режиме эта ячейка вызывает `/embeddings` и расходует средства. Первые координаты сами по себе человеку ничего не объясняют; интересна близость векторов.

In [7]:
embeddings = make_embeddings(settings, demo=DEMO)
samples = ["Библиотека выдаёт книги студентам.",
           "Академическая стипендия составляет 7000 рублей.",
           "Компьютеры можно бронировать на два часа."]
vectors = embeddings.embed_documents(samples)
query_vector = embeddings.embed_query("Как получить стипендию?")
print("Размерность:", len(vectors[0]))
print("Первые 8 координат:", vectors[0][:8])

def cosine(a, b):
    import math
    denominator = math.sqrt(sum(x*x for x in a) * sum(x*x for x in b))
    return sum(x*y for x, y in zip(a, b)) / denominator if denominator else 0

for text, vector in zip(samples, vectors):
    print(round(cosine(query_vector, vector), 4), text)

Размерность: 256
Первые 8 координат: [0.0, 0.0, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0]
0.0 Библиотека выдаёт книги студентам.
0.0 Академическая стипендия составляет 7000 рублей.
0.0 Компьютеры можно бронировать на два часа.


## 4. Сохраняем векторы и учитываем изменения
Chroma хранит текст, метаданные и векторы, SQLRecordManager — сведения, необходимые для пропуска неизменённых данных и удаления старых версий. В книге используется PGVector; локальный Chroma позволяет провести практикум без сервера PostgreSQL.

Здесь `cleanup="full"`: на вход подаётся **полный снимок каталога**, включая все источники. Это позволяет удалять записи исчезнувших файлов. В книге также показан `incremental`, который обрабатывает старые версии только переданных источников.

In [8]:
store, manager = open_store(settings, embeddings, ROOT / ".index-notebook", DEMO)
first_run = synchronize(chunks, store, manager)
second_run = synchronize(chunks, store, manager)
print("Первый запуск (если индекс уже есть, возможны skipped):", first_run)
print("Повтор:", second_run)
assert second_run["num_added"] == 0
print("Записей в Chroma:", store._collection.count())

Первый запуск (если индекс уже есть, возможны skipped): {'num_added': 9, 'num_updated': 0, 'num_skipped': 0, 'num_deleted': 0}
Повтор: {'num_added': 0, 'num_updated': 0, 'num_skipped': 9, 'num_deleted': 0}
Записей в Chroma: 9


## 5. Проверяем, что индекс находит контекст
Это проверка индекса, без генерации ответа. При cosine distance меньшее значение означает большую близость; это не уверенность и не вероятность ответа.

In [9]:
question = "Как получить стипендию?"
results = store.similarity_search_with_score(question, k=3)
for doc, distance in results:
    print(f"distance={distance:.4f}", doc.metadata)
    print(doc.page_content, "\n")

distance=0.9420 {'source': 'laboratory.txt', 'start_index': 0}
Компьютерная лаборатория

Лаборатория находится в корпусе Б, кабинет 204. Она открыта с 08:30 до 19:00 по будням. Для самостоятельной работы забронируйте компьютер через личный кабинет. Максимальная длительность одного бронирования — два часа.

На компьютерах установлены Python 3.12, Git и Visual Studio Code. Для практикума по LangChain создавайте отдельное виртуальное окружение. API-ключи храните в локальном файле .env, который не добавляют в Git. После работы выйдите из своей учетной записи. 

distance=0.9437 {'start_index': 0, 'source': 'exams.md'}
# Экзамены и пересдачи

## Основная сессия
Зимняя экзаменационная сессия проходит с 12 по 25 января. Расписание публикуют в личном кабинете не позднее чем за две недели до первого экзамена. На экзамен нужно принести студенческий билет.

## Пересдача
При неудовлетворительной оценке студент имеет право на две пересдачи. Первую принимает преподаватель, вторую — комиссия из трёх ч

## 6. Изменение и удаление в отдельном учебном индексе
Исходные файлы не меняются. Сначала индексируем две записи, затем меняем факт и исключаем второй источник из полного снимка. Реальный режим вызывает эмбеддинги новых версий.

In [10]:
from langchain_core.documents import Document
changes_store, changes_manager = open_store(
    settings, embeddings, ROOT / ".index-notebook-changes", DEMO,
)
version1 = [
    Document(page_content="Стипендия 7000 рублей.", metadata={"source": "stipend.txt"}),
    Document(page_content="Библиотека открыта в субботу.", metadata={"source": "library.txt"}),
]
print("Исходная версия:", synchronize(version1, changes_store, changes_manager))
print("Повтор:", synchronize(version1, changes_store, changes_manager))
version2 = [Document(page_content="Стипендия 9000 рублей.", metadata={"source": "stipend.txt"})]
print("Изменение + удаление:", synchronize(version2, changes_store, changes_manager))
assert changes_store._collection.count() == 1
print(changes_store.similarity_search("Стипендия", k=1)[0])

Исходная версия: {'num_added': 2, 'num_updated': 0, 'num_skipped': 0, 'num_deleted': 0}
Повтор: {'num_added': 0, 'num_updated': 0, 'num_skipped': 2, 'num_deleted': 0}
Изменение + удаление: {'num_added': 1, 'num_updated': 0, 'num_skipped': 0, 'num_deleted': 2}
page_content='Стипендия 9000 рублей.' metadata={'source': 'stipend.txt'}


## 7. MultiVectorRetriever: ищем по короткому представлению
Каждое резюме связано с оригиналом через `doc_id`. Векторами представлены резюме, пользователю возвращается полный исходный текст. В demo резюме — первые 220 символов; это не LLM-суммаризация.

В реальном режиме установите `RUN_MULTIVECTOR=True` после заполнения `CHAT_MODEL`. Это дополнительные вызовы чат-модели и эмбеддингов. Индекс данного примера хранится в памяти.

In [11]:
from chapter2.advanced import build_multivector
RUN_MULTIVECTOR = DEMO  # Для платного опыта включите явно.
if RUN_MULTIVECTOR:
    retriever, summaries = build_multivector(documents, embeddings, settings, demo=DEMO)
    print("Представление в индексе:", summaries[0])
    parents = retriever.invoke("Стипендия")
    for parent in parents:
        print("Оригинал:", parent.metadata, len(parent.page_content), "символов")
        print(parent.page_content[:500])

Представление в индексе: page_content='# Учебные данные
Все названия, суммы, даты, правила и контакты в этом каталоге вымышлены.
Этот файл тоже попадёт в индекс: загрузчик читает все .txt, .md и .pdf рекурсивно.
' metadata={'doc_id': '1b51714e-14ca-471d-a027-34a0e8c50366', 'source': 'README.md'}
Оригинал: {'source': 'scholarships.md'} 636 символов
# Стипендии учебного центра «Орбита»

## Академическая стипендия
Академическая стипендия составляет 7000 рублей в месяц. Её получают студенты очной формы обучения, сдавшие все экзамены на хорошо и отлично без академической задолженности. Заявление подают в учебный офис до 15 сентября. К заявлению прикладывают выписку с оценками за последний семестр.

## Социальная поддержка
Социальная стипендия составляет 10000 рублей в месяц и не зависит от оценок. Для оформления нужны заявление и справка о пра
Оригинал: {'source': 'library.txt'} 516 символов
Библиотека учебного центра «Орбита»

Библиотека работает по будням с 09:00 до 20:00, в субботу с 10:

## Задания
1. Найдите фрагмент с условиями выплаты стипендии и его `source`.
2. Сравните чанки 250/0 и 500/100: где факт потерял контекст?
3. Сравните реальный поиск по словам и по синонимам. Почему demo не подходит для этой оценки?
4. Что случится с исчезнувшим источником при `incremental`? Почему здесь выбран `full`?
5. Чем MultiVectorRetriever отличается от RAPTOR и ColBERT? См. `docs/lesson.md`.

Следующий этап, глава 3: передать найденный контекст в LLM и проверить достаточность источников.